<a href="https://colab.research.google.com/github/caladriusllc-code/caladrius-ml-learn/blob/learning-branch/cnn_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from functools import partial
import tensorflow as tf
from tensorflow import keras

In [ ]:
DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding="same", activation="relu", kernel_initializer="he_normal")
model = tf.keras.Sequential([
  DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]), tf.keras.layers.MaxPool2D(),
  DefaultConv2D(filters=128),
  DefaultConv2D(filters=128),
  tf.keras.layers.MaxPool2D(),
  DefaultConv2D(filters=256),
  DefaultConv2D(filters=256),
  tf.keras.layers.MaxPool2D(),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(units=128, activation="relu",
  kernel_initializer="he_normal"), tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Dense(units=64, activation="relu", kernel_initializer="he_normal"),
  tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Dense(units=10, activation="softmax")
])

In [ ]:
# implémenter un cnn à partir de zéro

DefaultConv2D = partial(tf.keras.layers.Conv2D,
                        kernel_size=3, strides=1,
                        padding="same",
                        kernel_initializer="he_normal",
                        use_bias=False)

class ResidualUnit(tf.keras.layers.Layer):

  def __init__(self, filters, strides=1, activation="relu", **kwargs):
    super().__init__(**kwargs)
    self.activation = tf.keras.activations.get(activation)
    self.main_layers = [
      DefaultConv2D(filters, strides=strides), tf.keras.layers.BatchNormalization(), self.activation,
      DefaultConv2D(filters), tf.keras.layers.BatchNormalization()
    ]
    self.skip_layers = [] if strides > 1:
    self.skip_layers = [
    DefaultConv2D(filters, kernel_size=1, strides=strides), tf.keras.layers.BatchNormalization()
    ]
  def call(self, inputs): Z = inputs
    for layer in self.main_layers: Z = layer(Z)
    skip_Z = inputs
    for layer in self.skip_layers:
    skip_Z = layer(skip_Z)
    return self.activation(Z + skip_Z)

In [ ]:
# implementer un Resnet-34

model = tf.keras.Sequential([
    DefaultConv2D(64, kernel_size=7, strides=2, input_shape=[224, 224, 3]),
    tf.keras.layer.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same"),
])
prev_filters = 64
for filter in filters:
  strides = 1 if prev_filters == filter else 2
  model.add(ResidualUnit(filter, strides=strides))
  prev_filters = filter

model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(10, activation="softmax"))
